# Load Data

In [17]:
# piterfm/fifa-football-world-cup

import kaggle

kaggle.api.authenticate()

kaggle.api.dataset_download_files('piterfm/fifa-football-world-cup',path='./sourcedata', unzip=True)
kaggle.api.dataset_download_files('cashncarry/fifaworldranking',path='./sourcedata', unzip=True)


Dataset URL: https://www.kaggle.com/datasets/piterfm/fifa-football-world-cup
Dataset URL: https://www.kaggle.com/datasets/cashncarry/fifaworldranking


# Transformation

## *Worldcup Matches* Table

### 1. Column Selection

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Get necessary columns
df = pd.read_csv('sourcedata/matches_1930_2022.csv')
df = df[['Year', 'Host', 'Date', 'Round', 'Attendance', 'Venue', 
          'home_team', 'home_score', 'home_penalty', 
          'away_team', 'away_score', 'away_penalty',
          'home_goal', 'home_own_goal', 'home_penalty_goal', 
          'away_goal', 'away_own_goal', 'away_penalty_goal']]


In [2]:
df.head(2)

,Year,Host,Date,Round,Attendance,Venue,home_team,home_score,home_penalty,away_team,away_score,away_penalty,home_goal,home_own_goal,home_penalty_goal,away_goal,away_own_goal,away_penalty_goal
0,2022,Qatar,2022-12-18,Final,88966,"Lusail Iconic Stadium, Lusail",Argentina,3,4.0,France,3,2.0,Ángel Di María · 36|Lionel Messi · 108,NaN,Lionel Messi (P) · 23,Kylian Mbappé · 81,NaN,Kylian Mbappé (P) · 80|Kylian Mbappé (P) · 118
1,2022,Qatar,2022-12-17,Third-place match,44137,"Khalifa International Stadium, Doha",Croatia,2,NaN,Morocco,1,NaN,Joško Gvardiol · 7|Mislav Oršić · 42,NaN,NaN,Achraf Dari · 9,NaN,NaN


### 2. Get Goal Minutes

In [8]:
# goal minutes
def extract_minutes(text):
    minute_strings = re.findall(r'\d+\+?\d*', str(text))
    result = []
    for m in minute_strings:
        if '+' in m:
            base, extra = map(int, m.split('+'))
            result.append(base + extra)
        else:
            result.append(int(m))
    return result

df['home_goal_min'] = df['home_goal'].apply(extract_minutes)
df['home_own_goal_min'] = df['home_own_goal'].apply(extract_minutes)
df['home_penalty_goal_min'] = df['home_penalty_goal'].apply(extract_minutes)
df['away_goal_min'] = df['away_goal'].astype(str).apply(extract_minutes)
df['away_own_goal_min'] = df['away_own_goal'].apply(extract_minutes)
df['away_penalty_goal_min'] = df['away_penalty_goal'].apply(extract_minutes)

df.at[282, 'away_goal_min'] = [10, 20] #adjustment
df.at[271, 'home_goal_min'] = [10, 98] #adjustment


df = df[['Year', 'Host', 'Date', 'Round', 'Attendance', 'Venue', 
          'home_team', 'home_score', 'home_penalty', 
          'away_team', 'away_score', 'away_penalty',
          'home_goal_min', 'home_own_goal_min', 'home_penalty_goal_min', 
          'away_goal_min', 'away_own_goal_min', 'away_penalty_goal_min']]


### 3. Column `match_win`, `penalyty_win`

In [10]:
# Match winner
df['match_win'] = df.apply(lambda row: 'home' if row['home_score'] > row['away_score']
                          else 'away' if row['home_score'] < row['away_score']
                          else 'draw',axis=1)
# Shootout winner
df['penalty_win'] = df.apply(
    lambda row: np.nan if pd.isna(row['home_penalty']) or pd.isna(row['away_penalty']) or row['home_penalty'] == 0 or row['away_penalty'] == 0
    else 'home' if row['home_penalty'] > row['away_penalty']
    else 'away' if row['home_penalty'] < row['away_penalty']
    else 'draw',
    axis=1
)


### 4. Column `first_goal`

In [11]:
# First Goal

import numpy as np

def safe_min(goal_lists):
    combined = []
    for lst in goal_lists:
        if isinstance(lst, list):
            combined += lst
    return min(combined) if combined else np.inf

def first_goal(row):
    if row['home_score'] + row['away_score'] == 0:
        return np.nan
    
    home_times = row['home_goal_min'] + row['home_own_goal_min'] + row['away_penalty_goal_min']
    away_times = row['away_goal_min'] + row['away_own_goal_min'] + row['away_penalty_goal_min']
    
    home_min = safe_min([home_times])
    away_min = safe_min([away_times])
    
    return 'home' if home_min < away_min else 'away'

df['firstGoal'] = df.apply(first_goal, axis=1)


In [15]:
data=[]
for i in range(0,len(df)):
    a = len(df['home_goal_min'][i])+len(df['home_own_goal_min'][i])+len(df['home_penalty_goal_min'][i])
    b = df['home_score'][i]
    c = len(df['away_goal_min'][i])+len(df['away_own_goal_min'][i])+len(df['away_penalty_goal_min'][i])
    d = df['away_score'][i]
    data.append(a==b and c==d)

false_indices = [i for i, val in enumerate(data) if val == False]
print(false_indices)


[]


### 5. Get Fifa Rank: `home_rank`, `away_rank`

In [18]:
# FIFA Rank

FR = pd.read_csv('sourcedata/fifa_ranking-2023-07-20.csv')

# Rank dates for each World Cup
rank_dates = {1994: '1994-06-14', 1998: '1998-05-20', 2002: '2002-05-15', 2006: '2006-05-17',
              2010: '2010-05-26', 2014: '2014-06-05', 2018: '2018-06-07', 2022: '2022-10-06'}

# Get Rank and Create DataFrame
all_data = []

for year, rank_date in rank_dates.items():
    teams = df[df['Year'] == year]['home_team'].tolist() + df[df['Year'] == year]['away_team'].tolist()
    teams = list(set(teams))  # Remove duplicates

    for team in teams:
        result = FR[(FR['rank_date'] == rank_date) & (FR['country_full'] == team)]['rank']
        rank = result.values[0] if not result.empty else None  # or np.nan
        all_data.append({'team': team, 'rank': rank, 'Year': year})


team_rank_df = pd.DataFrame(all_data)
team_rank_df = team_rank_df.sort_values(by=['Year', 'rank'])

# Adjustment
team_rank_df.loc[(team_rank_df['team'] == 'United States') & (team_rank_df['Year'] == 1994),'rank'] = 23
team_rank_df.loc[(team_rank_df['team'] == 'United States') & (team_rank_df['Year'] == 1998),'rank'] = 11
team_rank_df.loc[(team_rank_df['team'] == 'FR Yugoslavia') & (team_rank_df['Year'] == 1998),'rank'] = 8
team_rank_df.loc[(team_rank_df['team'] == 'United States') & (team_rank_df['Year'] == 2002),'rank'] = 13
team_rank_df.loc[(team_rank_df['team'] == 'Türkiye') & (team_rank_df['Year'] == 2002),'rank'] = 22
team_rank_df.loc[(team_rank_df['team'] == 'United States') & (team_rank_df['Year'] == 2006),'rank'] = 5
team_rank_df.loc[(team_rank_df['team'] == 'United States') & (team_rank_df['Year'] == 2010),'rank'] = 14
team_rank_df.loc[(team_rank_df['team'] == 'United States') & (team_rank_df['Year'] == 2014),'rank'] = 13
team_rank_df.loc[(team_rank_df['team'] == 'United States') & (team_rank_df['Year'] == 2022),'rank'] = 16

# team_rank_df[team_rank_df['rank'].isna()]

# Merge rank for home teams
df = df.merge(
    team_rank_df.rename(columns={'team': 'home_team', 'rank': 'home_rank'}),
    how='left',
    on=['home_team', 'Year']
)

# Merge rank for away teams
df = df.merge(
    team_rank_df.rename(columns={'team': 'away_team', 'rank': 'away_rank'}),
    how='left',
    on=['away_team', 'Year']
)



### 6. Get CSV

In [23]:
df.to_csv('worldcup_matches.csv', index=False)

## *Team Stats* table

### get stats by team (nation)

In [24]:
# Initialize an empty dictionary to store team records
team_records = {}

# Process each match
for _, row in df.iterrows():
    home_team = row['home_team']
    away_team = row['away_team']
    home_score = row['home_score']
    away_score = row['away_score']
    home_penalty = row['home_penalty']
    away_penalty = row['away_penalty']
    round_type = row['Round']
    year = row['Year']
    
    # Initialize team records if not exists
    for team in [home_team, away_team]:
        if team not in team_records:
            team_records[team] = {
                'matches_played': 0, 'win': 0, 'draw': 0, 'lose': 0,
                'shootout_played': 0, 'shootout_win': 0, 'shootout_lose': 0,
                'semifinal_played': 0, 'final_played': 0, 'champion': 0,
                'goal_scored': 0, 'goal_conceded': 0,
                'years_participated': set()
            }
    
    # Update participation years
    team_records[home_team]['years_participated'].add(year)
    team_records[away_team]['years_participated'].add(year)
    
    # Update matches taken
    team_records[home_team]['matches_played'] += 1
    team_records[away_team]['matches_played'] += 1
    
    # Update goals scored and conceded
    team_records[home_team]['goal_scored'] += home_score
    team_records[home_team]['goal_conceded'] += away_score
    team_records[away_team]['goal_scored'] += away_score
    team_records[away_team]['goal_conceded'] += home_score
    
    # Check if it's a semifinal or final
    if round_type in ['Semi-finals', 'Final stage']:
        team_records[home_team]['semifinal_played'] += 1
        team_records[away_team]['semifinal_played'] += 1
    elif round_type == 'Final':
        team_records[home_team]['final_played'] += 1
        team_records[away_team]['final_played'] += 1
    
    # Determine match outcome
    if row['match_win'] == 'home':
        team_records[home_team]['win'] += 1
        team_records[away_team]['lose'] += 1
        
        if round_type == 'Final':
            team_records[home_team]['champion'] += 1
    elif row['match_win'] == 'away':
        team_records[away_team]['win'] += 1
        team_records[home_team]['lose'] += 1
        
        if round_type == 'Final':
            team_records[away_team]['champion'] += 1
    else:  # draw
        team_records[home_team]['draw'] += 1
        team_records[away_team]['draw'] += 1
        
        # Check for penalty shootout
        if not pd.isna(home_penalty) and not pd.isna(away_penalty):
            
            # Count the shootout played for both teams
            team_records[home_team]['shootout_played'] += 1
            team_records[away_team]['shootout_played'] += 1
            
            if home_penalty > away_penalty:
                team_records[home_team]['shootout_win'] += 1
                team_records[away_team]['shootout_lose'] += 1
            
                if round_type == 'Final':
                    team_records[home_team]['champion'] += 1
            
            else:
                team_records[away_team]['shootout_win'] += 1
                team_records[home_team]['shootout_lose'] += 1
                
                if round_type == 'Final':
                    team_records[home_team]['champion'] += 1
            

# Convert the dictionary to a DataFrame
teamStat = pd.DataFrame.from_dict(team_records, orient='index')

# Calculate worldcup participation count
teamStat['worldcup_participation'] = teamStat['years_participated'].apply(len)
teamStat.drop('years_participated', axis=1, inplace=True)

# Reset index to make team a column
teamStat.reset_index(inplace=True)
teamStat.rename(columns={'index': 'team'}, inplace=True)

# Reorder columns
column_order = [
    'team', 'matches_played', 'win', 'draw', 'lose',
    'shootout_played', 'shootout_win', 'shootout_lose', 
    'semifinal_played', 'final_played', 'champion', 
    'goal_scored', 'goal_conceded', 
    'worldcup_participation'
]
teamStat = teamStat[column_order]

# Manually correct Uruguay's champion count for 1950 format
teamStat.loc[teamStat['team'] == 'Uruguay', 'champion'] = 2

# Sort by matches taken (descending)
teamStat.sort_values('matches_played', ascending=False, inplace=True)


### Standardize country name

In [25]:
# Define country groupings (modern name → historical names)
country_mapping = {
    'Germany': ['West Germany', 'Germany'],
    'Russia': ['Soviet Union', 'Russia'],
    'Serbia': ['FR Yugoslavia', 'Serbia and Montenegro', 'Serbia', 'Yugoslavia'],
    'Czech Republic': ['Czech Republic', 'Czechoslovakia'],
    # Add other mappings if needed (e.g., Slovakia)
}

# Function to replace historical names with modern ones
def standardize_country(team_name):
    for modern_name, historical_names in country_mapping.items():
        if team_name in historical_names:
            return modern_name
    return team_name  # If no mapping, keep original name

# Apply the standardization
teamStat['team_modern'] = teamStat['team'].apply(standardize_country)

# Aggregate stats by modern country (summing numeric columns)
numeric_cols = ['matches_played', 'win', 'draw', 'lose', 'goal_scored', 'goal_conceded', 
                'champion', 'final_played', 'semifinal_played','worldcup_participation']  # Add all relevant columns

teamStat_modern = teamStat.groupby('team_modern', as_index=False)[numeric_cols].sum()

teamStat_modern.sort_values('matches_played', ascending=False, inplace=True)



In [26]:
# Save the final data
teamStat_modern.to_csv('wc_team_stat.csv', index=False)

## *Penalty Shootout* Table

In [28]:
# Filter only matches with penalty shootouts
penalty_matches = df[df['home_penalty'].notna() | df['away_penalty'].notna()].copy()

# Define columns to keep from original data
metadata_cols = ['Year', 'Host', 'Round', 'home_team', 'away_team', 'home_penalty', 'away_penalty']

# Create home team entries
home_teams = penalty_matches[metadata_cols].copy()
home_teams['team_type'] = 'home_team'
home_teams.rename(columns={
    'home_team': 'team',
    'home_penalty': 'penalty_score',
    'away_penalty': 'opponent_score'
}, inplace=True)

# Create away team entries
away_teams = penalty_matches[metadata_cols].copy()
away_teams['team_type'] = 'away_team'
away_teams.rename(columns={
    'away_team': 'team',
    'away_penalty': 'penalty_score',
    'home_penalty': 'opponent_score'
}, inplace=True)

# Combine into one dataset
combined = pd.concat([home_teams, away_teams])

# Calculate shootout result
combined['result'] = combined.apply(
    lambda x: 'Win' if x['penalty_score'] > x['opponent_score'] 
              else 'Loss' if x['penalty_score'] < x['opponent_score'] 
              else 'Draw', axis=1
)

# Add context for visualization
combined['shootout_appearance'] = 1
combined['is_winner'] = (combined['result'] == 'Win').astype(int)

# Save to CSV with all metadata
output_cols = ['Year', 'Host', 'Round', 'team', 'team_type', 
               'penalty_score', 'opponent_score', 'result', 'shootout_appearance', 'is_winner']

df_PKmatches = combined[output_cols]


In [31]:
df_PKmatches.to_csv('shootout_analysis.csv', index=False)

In [32]:
df.head()

,Year,Host,Date,Round,Attendance,Venue,home_team,home_score,home_penalty,away_team,...,home_own_goal_min,home_penalty_goal_min,away_goal_min,away_own_goal_min,away_penalty_goal_min,match_win,penalty_win,firstGoal,home_rank,away_rank
0,2022,Qatar,2022-12-18,Final,88966,"Lusail Iconic Stadium, Lusail",Argentina,3,4.0,France,...,[],[23],[81],[],"[80, 118]",draw,home,home,3.0,4.0
1,2022,Qatar,2022-12-17,Third-place match,44137,"Khalifa International Stadium, Doha",Croatia,2,NaN,Morocco,...,[],[],[9],[],[],home,NaN,home,12.0,22.0
2,2022,Qatar,2022-12-14,Semi-finals,68294,"Al Bayt Stadium, Al Khor",France,2,NaN,Morocco,...,[],[],[],[],[],home,NaN,home,4.0,22.0
3,2022,Qatar,2022-12-13,Semi-finals,88966,"Lusail Iconic Stadium, Lusail",Argentina,3,NaN,Croatia,...,[],[34],[],[],[],home,NaN,home,3.0,12.0
4,2022,Qatar,2022-12-10,Quarter-finals,44198,"Al Thumama Stadium, ath-Thumāma",Morocco,1,NaN,Portugal,...,[],[],[],[],[],home,NaN,home,22.0,9.0
